In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash kailash-ml kailash-kaizen polars plotly gdown python-dotenv nest-asyncio

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP M5 inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated from shared/. Click the ▼ arrow on the left to hide.
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations


# ── kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model

# ── diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]

# ── ex_8.py ──
"""
Shared infrastructure for Exercise 8 — Reinforcement Learning.

Contains: CartPole setup, reward plotting helpers, ExperimentTracker/ModelRegistry
setup, custom environment base class, evaluation utilities.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
import random
from collections import deque
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import gymnasium as gym
from gymnasium import spaces

import polars as pl

from kailash.db import ConnectionManager
from kailash_ml import ModelVisualizer
from kailash_ml.engines.experiment_tracker import ExperimentTracker
from kailash_ml.engines.model_registry import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex8_reinforcement_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# CARTPOLE ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════


def make_cartpole() -> tuple[gym.Env, int, int]:
    """Create CartPole-v1 and return (env, obs_dim, n_actions)."""
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    print(f"CartPole-v1  obs_dim={obs_dim}  n_actions={n_actions}")
    return env, obs_dim, n_actions


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    conn = ConnectionManager("sqlite:///mlfp05_rl.db")
    await conn.initialize()

    tracker = ExperimentTracker(conn)
    exp_name = await tracker.create_experiment(
        name="m5_reinforcement_learning",
        description="RL algorithms: DQN and PPO on CartPole and business envs",
    )

    try:
        registry = ModelRegistry(conn)
        has_registry = True
    except Exception as e:
        registry = None
        has_registry = False
        print(f"  Note: ModelRegistry setup skipped ({e})")

    return conn, tracker, exp_name, registry, has_registry


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# REPLAY BUFFER — shared by DQN and custom env training
# ════════════════════════════════════════════════════════════════════════


class ReplayBuffer:
    """Fixed-size buffer storing (state, action, reward, next_state, done)."""

    def __init__(self, capacity: int = 10_000):
        self.buffer: deque = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(list(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# ════════════════════════════════════════════════════════════════════════
# DQN NETWORK — shared by DQN training and custom env training
# ════════════════════════════════════════════════════════════════════════


class DQN(nn.Module):
    """Deep Q-Network: maps state -> Q-value for each action."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION UTILITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_policy(env: gym.Env, policy_fn, n_episodes: int = 30) -> list[float]:
    """Evaluate a policy function over n_episodes. Returns list of total rewards."""
    eval_returns: list[float] = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=1000 + i)
        total = 0.0
        done = False
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            done = terminated or truncated
        eval_returns.append(total)
    return eval_returns


# ════════════════════════════════════════════════════════════════════════
# REWARD PLOTTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def moving_average(xs: list[float], window: int = 10) -> list[float]:
    """Smooth a time series with a rolling mean."""
    if len(xs) < window:
        return xs
    arr = np.asarray(xs, dtype=np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    return list(np.convolve(arr, kernel, mode="valid"))


def plot_reward_curve(
    viz: ModelVisualizer,
    rewards: list[float],
    title: str,
    filename: str,
    window: int = 20,
    x_label: str = "Episode",
    y_label: str = "Reward",
) -> None:
    """Plot a reward curve with moving average and save to HTML."""
    metrics = {
        f"{title} reward": rewards,
        f"{title} moving avg ({window})": moving_average(rewards, window),
    }
    fig = viz.training_history(metrics=metrics, x_label=x_label, y_label=y_label)
    out_path = OUTPUT_DIR / filename
    fig.write_html(str(out_path))
    print(f"  Saved: {out_path}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Register a single RL policy network in ModelRegistry."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    metric_specs = [MetricSpec(name=k, value=v) for k, v in metrics_dict.items()]
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metric_specs,
    )
    print(f"  Registered {name}: version={version.version}")
    return version


def register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Sync wrapper for RL model registration."""
    return asyncio.run(_register_rl_model(registry, name, model, metrics_dict))



# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 8.3: Custom Gymnasium Environments for Business
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Explain why custom environments matter: RL is only as good as the
#     simulation it trains on
#   - Build 5 Gymnasium-compliant environments for real Singapore
#     business problems
#   - Design observation spaces, action spaces, reward functions, and
#     episode termination logic for each domain
#   - Train DQN on a custom environment and register the trained policy
#     in the ModelRegistry
#   - Visualise environment state trajectories and action distributions
#   - Evaluate learned policies against rule-based baselines
#
# PREREQUISITES: M5/ex_8/01_dqn.py and M5/ex_8/02_ppo.py.
# ESTIMATED TIME: ~40 min
# DATASETS: No static dataset — the environments ARE the data source.
#
# TASKS:
#   1. Theory: why custom environments are the foundation of applied RL
#   2. Build 5 business-themed Gymnasium environments
#   3. Train DQN on ChurnPrevention, register in ModelRegistry
#   4. Visualise: environment state trajectories, action distributions,
#      learned policy vs baseline
#   5. Apply: each environment IS the application — evaluate ChurnPrevention
#      with churn rate reduction and revenue impact metrics
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

import gymnasium as gym
from gymnasium import spaces

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

    DQN,
    OUTPUT_DIR,
    ReplayBuffer,
    device,
    evaluate_policy,
    moving_average,
    register_rl_model,
    setup_engines,
)
from kailash_ml import ModelVisualizer



## TASK 1 — Theory: Why Custom Environments Matter


In [ ]:
# CartPole is a toy. In the real world, RL agents don't balance poles —
# they make BUSINESS DECISIONS: pricing, inventory, resource allocation,
# customer retention, portfolio management.
#
# The environment IS the problem definition. Get it wrong, and even a
# perfect RL algorithm will learn the wrong thing. Building a good
# environment requires:
#
#   1. STATE: What information does the decision-maker observe?
#      Too little -> agent can't learn. Too much -> curse of dimensionality.
#
#   2. ACTIONS: What can the decision-maker do?
#      Too few -> can't express optimal behaviour. Too many -> slow learning.
#
#   3. REWARD: What defines success?
#      This is the HARDEST part. Reward shaping is an art:
#      - Too sparse (reward only at episode end) -> agent can't learn
#      - Too dense (reward every step) -> agent may "hack" the reward
#      - Misaligned (reward proxy, not true objective) -> Goodhart's Law
#
#   4. DYNAMICS: How does the world respond to actions?
#      Must be realistic enough to transfer to the real system.
#
# Each environment below models a REAL business problem with realistic
# dynamics calibrated to Singapore market conditions.

print("=" * 70)
print("  TASK 1: Custom Environments — The Foundation of Applied RL")
print("=" * 70)

conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build 5 Business-Themed Gymnasium Environments


In [ ]:
print("\n" + "=" * 70)
print("  TASK 2: Build 5 Custom Environments")
print("=" * 70)



### Environment 1: Customer Churn Prevention (Singtel / StarHub)


SCENARIO: You manage the retention team at a Singapore telecom
    (think Singtel or StarHub). Each day you observe a customer's health
    metrics and decide whether/how to intervene.

    State (4,): [satisfaction_score, usage_frequency, months_active, support_tickets]
      All normalised [0, 1]. satisfaction decays naturally; tickets accumulate.

    Actions (4):
      0 = do nothing (free)
      1 = discount offer (costs $1, boosts satisfaction + usage)
      2 = proactive support call (costs $0.50, reduces tickets + boosts satisfaction)
      3 = feature upgrade (costs $1.50, boosts usage)

    Reward: +10 for retaining a customer through the month, -5 for churn,
            +1 per day customer stays, minus intervention cost.

    Episode: 30 steps (one month of daily decisions for one customer).


In [ ]:
class ChurnPreventionEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(4)
        self.max_steps = 30
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.np_random.uniform(0.2, 0.8, size=(4,)).astype(np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        satisfaction, usage, tenure, tickets = self.state

        # TODO: Implement action effects on customer state
        # Action 0 = do nothing (cost 0)
        # Action 1 = discount: satisfaction +0.1, usage +0.05, cost 1.0
        # Action 2 = support call: tickets -0.15, satisfaction +0.05, cost 0.5
        # Action 3 = feature upgrade: usage +0.1, cost 1.5
        # Hint: clip satisfaction and usage to [0, 1] using min()
        intervention_cost = 0.0
        if action == 1:  # discount
            satisfaction = # TODO: min(1.0, satisfaction + 0.1)
            usage = # TODO: min(1.0, usage + 0.05)
            intervention_cost = 1.0
        elif action == 2:  # support call
            tickets = # TODO: max(0.0, tickets - 0.15)
            satisfaction = # TODO: min(1.0, satisfaction + 0.05)
            intervention_cost = 0.5
        elif action == 3:  # feature upgrade
            usage = # TODO: min(1.0, usage + 0.1)
            intervention_cost = 1.5

        # TODO: Natural drift — satisfaction decays, tickets accumulate
        # Hint: satisfaction -= 0.02 + noise, usage drifts slightly, tickets grow
        satisfaction = # TODO: max(0.0, satisfaction - 0.02 + np_random.normal(0, 0.02))
        usage = # TODO: max(0.0, min(1.0, usage - 0.01 + np_random.normal(0, 0.02)))
        tickets = # TODO: max(0.0, min(1.0, tickets + 0.02 + np_random.normal(0, 0.01)))
        tenure = min(1.0, tenure + 1.0 / self.max_steps)

        self.state = np.array([satisfaction, usage, tenure, tickets], dtype=np.float32)

        # TODO: Compute churn probability and determine if customer churns
        # Hint: churn_prob = max(0.0, 0.3 - satisfaction * 0.4 + tickets * 0.3)
        # Hint: churned = self.np_random.random() < churn_prob
        churn_prob = # TODO
        churned = # TODO

        if churned:
            reward = -5.0
            terminated = True
        else:
            reward = 1.0 - intervention_cost
            terminated = False

        truncated = self.step_count >= self.max_steps
        if truncated and not terminated:
            reward += 10.0  # bonus for retaining customer through the month

        return self.state.copy(), reward, terminated, truncated, {}



### Environment 2: Portfolio Rebalancing (hedge fund)


SCENARIO: You manage a Singapore-domiciled fund investing in three
    asset classes: SGX equities, Singapore government bonds, and cash (SGD).

    State (6,): [weight_stocks, weight_bonds, weight_cash,
                 market_volatility, interest_rate, momentum]
    Actions (27): 3^3 = all combinations of (decrease/hold/increase) for
                  each of the 3 assets. Weights are re-normalised after.
    Reward: portfolio_return - 0.5 * volatility_penalty - transaction_cost
    Episode: 24 steps (monthly rebalancing over 2 years).


Decode flat action to per-asset decisions: 0=decrease, 1=hold, 2=increase.


In [ ]:
class PortfolioRebalancingEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(6,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(27)
        self.max_steps = 24
        self.state = None
        self.step_count = 0

    def _decode_action(self, action: int) -> list[int]:
        return [(action // (3**i)) % 3 for i in range(3)]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        weights = np.array([0.4, 0.4, 0.2], dtype=np.float32)
        market = self.np_random.uniform(0.2, 0.5)
        rate = self.np_random.uniform(0.3, 0.6)
        momentum = 0.5
        self.state = np.concatenate([weights, [market, rate, momentum]]).astype(
            np.float32
        )
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        weights = self.state[:3].copy()
        market_vol, interest, momentum = self.state[3], self.state[4], self.state[5]

        # TODO: Decode action and apply weight shifts
        # Hint: decisions = self._decode_action(action)
        # Hint: shifts = [[-0.05, 0.0, 0.05][d] for d in decisions]
        # Hint: transaction_cost = 0.005 * sum(abs(shifts))
        # Hint: weights = clip(weights + shifts, 0, 1) then normalise
        decisions = # TODO
        shifts = # TODO
        transaction_cost = # TODO
        weights = np.clip(weights + np.array(shifts, dtype=np.float32), 0.0, 1.0)
        weights = weights / (weights.sum() + 1e-8)

        # TODO: Simulate asset returns
        # Hint: stock_return = np_random.normal(0.01 + momentum * 0.02, market_vol * 0.1)
        # Hint: bond_return = np_random.normal(interest * 0.005, 0.02)
        # Hint: cash_return = 0.001
        # Hint: portfolio_return = dot(weights, [stock, bond, cash])
        stock_return = # TODO
        bond_return = # TODO
        cash_return = 0.001
        asset_returns = np.array([stock_return, bond_return, cash_return])
        portfolio_return = # TODO

        vol_penalty = market_vol * 0.05 * weights[0]
        reward = portfolio_return - vol_penalty - transaction_cost

        market_vol = np.clip(market_vol + self.np_random.normal(0, 0.03), 0.1, 0.9)
        interest = np.clip(interest + self.np_random.normal(0, 0.02), 0.1, 0.9)
        momentum = np.clip(momentum + self.np_random.normal(0, 0.1), 0.0, 1.0)

        self.state = np.concatenate([weights, [market_vol, interest, momentum]]).astype(
            np.float32
        )
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 3: Queue Management (Changi Airport)


SCENARIO: You manage immigration counter staffing at Changi Airport.
    Flights arrive in waves; you redistribute staff across three zones
    (T1, T2, T3) every 30 minutes.

    State (6,): [queue_t1, queue_t2, queue_t3, staff_t1, staff_t2, staff_t3]
      Queues normalised by capacity; staff normalised by total headcount.
    Actions (7): 6 shift pairs (move staff between zones) + do nothing
    Reward: -wait_time_penalty + service_bonus - reallocation_cost
    Episode: 48 steps (24-hour shift in 30-min intervals).


In [ ]:
class QueueManagementEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(6,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(7)
        self.max_steps = 48
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        queues = self.np_random.uniform(0.1, 0.3, size=3).astype(np.float32)
        staff = np.array([0.33, 0.33, 0.34], dtype=np.float32)
        self.state = np.concatenate([queues, staff]).astype(np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        queues = self.state[:3].copy()
        staff = self.state[3:].copy()

        # TODO: Reallocate staff based on action (0-5 = shift pairs, 6 = do nothing)
        # Hint: shift_pairs = [(0,1),(0,2),(1,0),(1,2),(2,0),(2,1)]
        # Hint: if action < 6: move 0.08 staff from src to dst, cost = 0.15
        shift_pairs = [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]
        realloc_cost = 0.0
        if action < 6:
            src, dst = shift_pairs[action]
            amount = min(0.08, staff[src])
            staff[src] -= amount
            staff[dst] += amount
            realloc_cost = 0.15

        # Flight arrival waves (Changi pattern: peaks at 6am, 12pm, 6pm, 11pm)
        half_hour = self.step_count % 48
        hour = half_hour / 2.0
        wave_t1 = 0.15 * np.exp(-0.5 * ((hour - 6) / 2) ** 2) + 0.1 * np.exp(
            -0.5 * ((hour - 18) / 2) ** 2
        )
        wave_t2 = 0.12 * np.exp(-0.5 * ((hour - 12) / 2) ** 2) + 0.08 * np.exp(
            -0.5 * ((hour - 23) / 2) ** 2
        )
        wave_t3 = 0.1 * np.exp(-0.5 * ((hour - 8) / 2) ** 2) + 0.12 * np.exp(
            -0.5 * ((hour - 20) / 2) ** 2
        )

        arrivals = np.array([wave_t1, wave_t2, wave_t3]) + self.np_random.normal(
            0, 0.02, size=3
        )
        arrivals = np.clip(arrivals, 0, 1)

        # TODO: Queue dynamics — arrivals add, staff serving removes
        # Hint: service_rate = staff * 0.3
        # Hint: queues = clip(queues + arrivals - service_rate, 0, 1)
        service_rate = # TODO
        queues = # TODO

        # TODO: Compute reward
        # Hint: avg_queue = mean(queues), max_queue = max(queues)
        # Hint: wait_penalty = 2.0 * avg_queue + 3.0 * max_queue
        # Hint: service_bonus = 1.0 if max_queue < 0.3 else 0.0
        avg_queue = # TODO
        max_queue = # TODO
        wait_penalty = # TODO
        service_bonus = # TODO
        reward = service_bonus - wait_penalty - realloc_cost

        self.state = np.concatenate([queues, staff]).astype(np.float32)
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 4: Energy Trading (SP Group)


SCENARIO: You manage the trading desk at SP Group. Every hour you
    decide whether to buy, sell, or hold electricity based on price
    forecasts, current reserves, and demand patterns.

    State (5,): [spot_price, reserve_level, demand_forecast, solar_output, time_of_day]
    Actions (5): 0=sell_large, 1=sell_small, 2=hold, 3=buy_small, 4=buy_large
    Reward: trading_profit - reserve_penalty
    Episode: 168 steps (hourly decisions for one week).


In [ ]:
class EnergyTradingEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(5)
        self.max_steps = 168
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array(
            [0.5, 0.5, 0.4, 0.0, 0.0], dtype=np.float32  # midnight start
        )
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        price, reserve, demand, solar, time_of_day = self.state

        # TODO: Execute trading action
        # Hint: trade_amounts = [-0.15, -0.05, 0.0, 0.05, 0.15]
        # Hint: trade_cost = abs(trade) * 0.01
        # If buying (trade > 0): reserve += trade, pnl = -trade * price
        # If selling (trade < 0): actual_sell = min(-trade, reserve), pnl = actual_sell * price
        trade_amounts = [-0.15, -0.05, 0.0, 0.05, 0.15]
        trade = trade_amounts[action]
        trade_cost = abs(trade) * 0.01

        if trade > 0:
            reserve = # TODO
            trading_pnl = # TODO
        elif trade < 0:
            actual_sell = # TODO
            reserve = # TODO
            trading_pnl = # TODO
        else:
            trading_pnl = 0.0

        # Consume reserves to meet demand
        consumption = demand * 0.08
        unmet = max(0.0, consumption - reserve)
        reserve = max(0.0, reserve - consumption)

        # Penalties
        reserve_penalty = 3.0 * unmet
        low_reserve_penalty = 1.0 if reserve < 0.15 else 0.0

        reward = trading_pnl - trade_cost - reserve_penalty - low_reserve_penalty

        # Advance time
        time_of_day = (time_of_day + 1.0 / 24.0) % 1.0
        hour = time_of_day * 24

        # Price dynamics (Singapore: peaks at 12-3pm and 7-9pm)
        afternoon_peak = 0.2 * np.exp(-0.5 * ((hour - 14) / 2) ** 2)
        evening_peak = 0.15 * np.exp(-0.5 * ((hour - 20) / 2) ** 2)
        price = np.clip(
            0.3 + afternoon_peak + evening_peak + self.np_random.normal(0, 0.05),
            0.1,
            1.0,
        )

        # Solar output (tropical: peaks at noon, zero at night)
        if 7 <= hour <= 19:
            solar = np.clip(
                0.5 * np.sin(np.pi * (hour - 7) / 12) + self.np_random.normal(0, 0.05),
                0.0,
                1.0,
            )
        else:
            solar = 0.0
        reserve = min(1.0, reserve + solar * 0.03)

        # Demand forecast
        demand = np.clip(
            0.3
            + afternoon_peak * 0.8
            + evening_peak * 0.6
            + self.np_random.normal(0, 0.04),
            0.1,
            1.0,
        )

        self.state = np.array(
            [price, reserve, demand, solar, time_of_day], dtype=np.float32
        )
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Environment 5: Traffic Signal Optimisation (LTA)


SCENARIO: You manage a 4-way intersection for the Land Transport
    Authority (LTA). Every cycle (90 seconds) you allocate green time
    between the north-south and east-west directions.

    State (4,): [queue_ns, queue_ew, flow_ns, flow_ew]
      Queues = vehicles waiting; flow = vehicles per minute arriving.
    Actions (5): green time allocation to NS direction
      0=20%, 1=35%, 2=50%, 3=65%, 4=80% (EW gets the remainder)
    Reward: -total_wait_time - queue_overflow_penalty
    Episode: 60 steps (90 minutes of signal cycles, ~1.5 hour rush).


In [ ]:
class TrafficSignalEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(4,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(5)
        self.max_steps = 60
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([0.2, 0.2, 0.4, 0.4], dtype=np.float32)
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        queue_ns, queue_ew, flow_ns, flow_ew = self.state

        # TODO: Allocate green time and compute service
        # Hint: ns_green_fraction = [0.20, 0.35, 0.50, 0.65, 0.80][action]
        # Hint: ew_green_fraction = 1.0 - ns_green_fraction
        # Hint: ns_served = min(queue_ns, ns_green_fraction * 0.5)
        # Hint: ew_served = min(queue_ew, ew_green_fraction * 0.5)
        ns_green_fraction = # TODO
        ew_green_fraction = # TODO

        ns_served = # TODO
        ew_served = # TODO
        queue_ns = max(0.0, queue_ns - ns_served)
        queue_ew = max(0.0, queue_ew - ew_served)

        # New arrivals (rush hour pattern)
        progress = self.step_count / self.max_steps
        rush_factor = 1.0 + 0.5 * np.sin(np.pi * progress)

        flow_ns = np.clip(0.3 * rush_factor + self.np_random.normal(0, 0.05), 0.1, 1.0)
        flow_ew = np.clip(0.35 * rush_factor + self.np_random.normal(0, 0.05), 0.1, 1.0)

        queue_ns = np.clip(queue_ns + flow_ns * 0.15, 0.0, 1.0)
        queue_ew = np.clip(queue_ew + flow_ew * 0.15, 0.0, 1.0)

        # Reward
        total_wait = queue_ns + queue_ew
        overflow_penalty = 2.0 * max(0.0, queue_ns - 0.8) + 2.0 * max(
            0.0, queue_ew - 0.8
        )
        reward = -total_wait - overflow_penalty

        self.state = np.array([queue_ns, queue_ew, flow_ns, flow_ew], dtype=np.float32)
        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}



### Validate all environments against Gymnasium API


In [ ]:
env_classes = [
    ("ChurnPrevention", ChurnPreventionEnv),
    ("PortfolioRebalancing", PortfolioRebalancingEnv),
    ("QueueManagement", QueueManagementEnv),
    ("EnergyTrading", EnergyTradingEnv),
    ("TrafficSignal", TrafficSignalEnv),
]
for env_name, env_cls in env_classes:
    test_env = env_cls()
    obs, info = test_env.reset(seed=42)
    assert (
        obs.shape == test_env.observation_space.shape
    ), f"{env_name}: obs shape mismatch"
    obs2, reward, terminated, truncated, info = test_env.step(
        test_env.action_space.sample()
    )
    assert (
        obs2.shape == test_env.observation_space.shape
    ), f"{env_name}: step obs shape mismatch"
    assert isinstance(reward, float), f"{env_name}: reward should be float"
    print(
        f"  {env_name}: obs={obs.shape}, actions={test_env.action_space.n}, reward={reward:.3f}"
    )
    test_env.close()



### Checkpoint 1


In [ ]:
# INTERPRETATION: Each environment models a real business decision:
# - ChurnPrevention: when to intervene (cost of action vs cost of losing customer)
# - PortfolioRebalancing: asset allocation (risk-return tradeoff, transaction costs)
# - QueueManagement: staff allocation (service quality vs disruption cost)
# - EnergyTrading: buy/sell timing (price forecasting, reserve management)
# - TrafficSignal: green time allocation (competing demands, queue overflow)
# All follow the Gymnasium API: reset() -> (obs, info), step(a) -> (obs, r, term, trunc, info)
print("--- Checkpoint 1 passed --- all 5 custom environments validated\n")



## TASK 3 — Train DQN on ChurnPrevention, register in ModelRegistry


Train DQN on ChurnPrevention under a tracker.run(...) context.


In [ ]:
print("=" * 70)
print("  TASK 3: Train DQN on ChurnPrevention Environment")
print("=" * 70)

churn_env = ChurnPreventionEnv()
churn_obs_dim = churn_env.observation_space.shape[0]
churn_n_actions = churn_env.action_space.n

churn_dqn = DQN(churn_obs_dim, churn_n_actions).to(device)
churn_target = DQN(churn_obs_dim, churn_n_actions).to(device)
churn_target.load_state_dict(churn_dqn.state_dict())
churn_target.eval()

churn_opt = torch.optim.Adam(churn_dqn.parameters(), lr=1e-3)
churn_replay = ReplayBuffer(capacity=10_000)
churn_rewards_hist: list[float] = []
churn_actions_hist: list[list[int]] = []  # track action distribution per episode


async def _train_churn_dqn_async():
    churn_epsilon = 1.0

    async with tracker.run(
        experiment_name=exp_name, run_name="dqn_churn_prevention"
    ) as ctx:
        await ctx.log_params(
            {
                "algorithm": "DQN",
                "environment": "ChurnPrevention",
                "gamma": "0.99",
                "episodes": "150",
            }
        )

        print("== Training DQN on ChurnPrevention ==")
        for ep in range(150):
            state, _ = churn_env.reset(seed=ep)
            total_reward = 0.0
            done = False
            ep_actions: list[int] = []

            while not done:
                # TODO: Epsilon-greedy action selection (same pattern as 01_dqn.py)
                # Hint: random action with probability churn_epsilon, else argmax of churn_dqn
                if random.random() < churn_epsilon:
                    action = # TODO
                else:
                    with torch.no_grad():
                        s_t = torch.tensor(state, dtype=torch.float32, device=device)
                        action = # TODO

                ep_actions.append(action)
                next_state, reward, terminated, truncated, _ = churn_env.step(action)
                done = terminated or truncated
                churn_replay.push(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward

                # TODO: Train on minibatch when replay has enough samples
                # Hint: same DQN training pattern — sample, compute Q-values,
                #   compute targets with churn_target, MSE loss, backprop
                if len(churn_replay) >= 300:
                    s_b, a_b, r_b, ns_b, d_b = churn_replay.sample(64)
                    q_vals = # TODO: churn_dqn(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
                    with torch.no_grad():
                        next_q = # TODO: churn_target(ns_b).max(dim=1).values
                        targets = # TODO: r_b + 0.99 * next_q * (1.0 - d_b)
                    loss = # TODO: F.mse_loss(q_vals, targets)
                    churn_opt.zero_grad()
                    loss.backward()
                    churn_opt.step()

            churn_epsilon = max(0.01, churn_epsilon * 0.99)
            if (ep + 1) % 10 == 0:
                churn_target.load_state_dict(churn_dqn.state_dict())

            churn_rewards_hist.append(total_reward)
            churn_actions_hist.append(ep_actions)
            await ctx.log_metric("episode_reward", total_reward, step=ep)

            if (ep + 1) % 30 == 0:
                avg_30 = float(np.mean(churn_rewards_hist[-30:]))
                print(
                    f"  ep {ep+1:3d}  reward={total_reward:6.1f}  "
                    f"avg30={avg_30:6.1f}  eps={churn_epsilon:.3f}"
                )

        await ctx.log_metric(
            "final_avg_reward", float(np.mean(churn_rewards_hist[-30:]))
        )


await _train_churn_dqn_async()

# Register trained model
if has_registry:
    register_rl_model(
        registry,
        "m5_dqn_churn_prevention",
        churn_dqn,
        {
            "avg_reward_last30": float(np.mean(churn_rewards_hist[-30:])),
            "episodes_trained": 150.0,
        },
    )



### Checkpoint 2


In [ ]:
assert len(churn_rewards_hist) == 150, "Churn DQN should train for 150 episodes"
print("--- Checkpoint 2 passed --- DQN trained on ChurnPrevention\n")



## TASK 4 — Visualise: state trajectories, action distributions,
          learned policy vs baseline


In [ ]:
print("=" * 70)
print("  TASK 4: Visualise Custom Environment Behaviour")
print("=" * 70)

viz = ModelVisualizer()



### Plot 1: ChurnPrevention training reward curve


In [ ]:
# TODO: Plot training reward curve with moving average using viz.training_history()
fig1 = # TODO
fig1.write_html(str(OUTPUT_DIR / "03_churn_training_curve.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_training_curve.html'}")



### Plot 2: Action distribution evolution


In [ ]:
# Compare action distributions: early training vs late training
action_names = ["Nothing", "Discount", "Support Call", "Feature Upgrade"]

early_actions = []
for ep_acts in churn_actions_hist[:20]:
    early_actions.extend(ep_acts)
late_actions = []
for ep_acts in churn_actions_hist[-20:]:
    late_actions.extend(ep_acts)

early_counts = [early_actions.count(i) / max(len(early_actions), 1) for i in range(4)]
late_counts = [late_actions.count(i) / max(len(late_actions), 1) for i in range(4)]

# TODO: Create grouped bar chart comparing early vs late action distributions
# Hint: go.Figure with two go.Bar traces — one for early, one for late
fig2 = # TODO
fig2.write_html(str(OUTPUT_DIR / "03_churn_action_distribution.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_action_distribution.html'}")
# INTERPRETATION: Early training shows near-uniform action distribution
# (random exploration). Late training shows the agent has learned WHEN
# to intervene — it should use "nothing" when the customer is healthy
# and targeted interventions when churn risk is high.



### Plot 3: State trajectory of a single episode (trained agent)


In [ ]:
churn_env_viz = ChurnPreventionEnv()
state, _ = churn_env_viz.reset(seed=99)
trajectory = {"step": [], "satisfaction": [], "usage": [], "tickets": [], "action": []}

done = False
step = 0
while not done:
    with torch.no_grad():
        s_t = torch.tensor(state, dtype=torch.float32, device=device)
        action = int(churn_dqn(s_t).argmax().item())
    trajectory["step"].append(step)
    trajectory["satisfaction"].append(float(state[0]))
    trajectory["usage"].append(float(state[1]))
    trajectory["tickets"].append(float(state[3]))
    trajectory["action"].append(action_names[action])
    state, _, terminated, truncated, _ = churn_env_viz.step(action)
    done = terminated or truncated
    step += 1

traj_df = pl.DataFrame(trajectory)

# TODO: Create a 2-row subplot — row 1: state trajectories (satisfaction, usage, tickets)
#   row 2: action timeline as coloured bars
# Hint: make_subplots(rows=2, cols=1, shared_xaxes=True)
# Hint: add go.Scatter traces for satisfaction/usage/tickets in row 1
# Hint: add go.Bar traces for action colours in row 2
fig3 = # TODO
fig3.write_html(str(OUTPUT_DIR / "03_churn_episode_trajectory.html"))
print(f"  Saved: {OUTPUT_DIR / '03_churn_episode_trajectory.html'}")

churn_env_viz.close()



### Checkpoint 3


In [ ]:
assert Path(OUTPUT_DIR / "03_churn_training_curve.html").exists()
assert Path(OUTPUT_DIR / "03_churn_action_distribution.html").exists()
assert Path(OUTPUT_DIR / "03_churn_episode_trajectory.html").exists()
print("--- Checkpoint 3 passed --- environment visualisations generated\n")



## TASK 5 — Apply: Evaluate ChurnPrevention with business metrics


In [ ]:
print("=" * 70)
print("  TASK 5: Business Impact — ChurnPrevention Policy Evaluation")
print("=" * 70)



### Baseline: always do nothing


In [ ]:
def do_nothing_policy(state):
    return 0



### Baseline: always discount (aggressive retention)


In [ ]:
def always_discount_policy(state):
    return 1



### Learned DQN policy


Evaluate a churn policy and return detailed metrics.


In [ ]:
def dqn_churn_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device)
        return int(churn_dqn(s).argmax().item())


def evaluate_churn_policy(env_cls, policy_fn, n_episodes):
    results = {
        "rewards": [],
        "churned": [],
        "steps_survived": [],
        "intervention_count": [],
    }
    for i in range(n_episodes):
        env = env_cls()
        state, _ = env.reset(seed=2000 + i)
        total_reward = 0.0
        interventions = 0
        done = False
        steps = 0
        while not done:
            action = policy_fn(state)
            if action > 0:
                interventions += 1
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
            steps += 1
        results["rewards"].append(total_reward)
        results["churned"].append(terminated)
        results["steps_survived"].append(steps)
        results["intervention_count"].append(interventions)
        env.close()
    return results


# Run 100 episodes for statistical significance
n_eval = 100
nothing_eval = evaluate_churn_policy(ChurnPreventionEnv, do_nothing_policy, n_eval)
discount_eval = evaluate_churn_policy(
    ChurnPreventionEnv, always_discount_policy, n_eval
)
dqn_eval = evaluate_churn_policy(ChurnPreventionEnv, dqn_churn_policy, n_eval)

# Business metrics
print(
    f"\n  {'Policy':<25} {'Churn Rate':>12} {'Avg Reward':>12} {'Interventions':>14} {'Avg Survival':>14}"
)
print(f"  {'-'*77}")
for name, results in [
    ("Do Nothing", nothing_eval),
    ("Always Discount", discount_eval),
    ("DQN Learned", dqn_eval),
]:
    churn_rate = sum(results["churned"]) / len(results["churned"]) * 100
    avg_reward = np.mean(results["rewards"])
    avg_interventions = np.mean(results["intervention_count"])
    avg_survival = np.mean(results["steps_survived"])
    print(
        f"  {name:<25} {churn_rate:>10.1f}% {avg_reward:>12.1f} {avg_interventions:>14.1f} {avg_survival:>12.1f}d"
    )

# Revenue impact calculation
monthly_revenue_per_customer = 50.0  # SGD (typical telecom ARPU)
intervention_cost_per_action = 5.0  # SGD average

for name, results in [
    ("Do Nothing", nothing_eval),
    ("Always Discount", discount_eval),
    ("DQN Learned", dqn_eval),
]:
    retention_rate = 1.0 - sum(results["churned"]) / len(results["churned"])
    avg_interventions = np.mean(results["intervention_count"])
    retained_revenue = retention_rate * monthly_revenue_per_customer
    total_cost = avg_interventions * intervention_cost_per_action
    net_value = retained_revenue - total_cost
    print(f"\n  {name}:")
    print(f"    Retention rate: {retention_rate*100:.1f}%")
    print(f"    Revenue retained: SGD {retained_revenue:.2f}/customer/month")
    print(f"    Intervention cost: SGD {total_cost:.2f}/customer/month")
    print(f"    Net value: SGD {net_value:.2f}/customer/month")



### Comparison visualisation


In [ ]:
all_rewards = nothing_eval["rewards"] + discount_eval["rewards"] + dqn_eval["rewards"]
all_labels = (
    ["Do Nothing"] * n_eval + ["Always Discount"] * n_eval + ["DQN Learned"] * n_eval
)
eval_df = pl.DataFrame({"Policy": all_labels, "Monthly Reward": all_rewards})
# TODO: Create box plot comparing the three policies
# Hint: viz.box_plot(eval_df, "Monthly Reward", group_by="Policy")
fig_eval = # TODO
fig_eval.write_html(str(OUTPUT_DIR / "03_churn_business_impact.html"))
print(f"\n  Saved: {OUTPUT_DIR / '03_churn_business_impact.html'}")

# INTERPRETATION: The DQN learns to intervene SELECTIVELY — only when
# churn risk is high, and with the most cost-effective action. "Always
# Discount" has good retention but high cost. "Do Nothing" has low cost
# but high churn. The DQN finds the sweet spot: high retention, moderate
# cost, maximum net value per customer.

churn_env.close()



### Checkpoint 4


In [ ]:
assert len(nothing_eval["rewards"]) == n_eval
assert len(dqn_eval["rewards"]) == n_eval
assert Path(OUTPUT_DIR / "03_churn_business_impact.html").exists()
print("--- Checkpoint 4 passed --- business impact evaluation complete\n")


# Clean up
asyncio.run(conn.close())



## REFLECTION


[x] Built 5 Gymnasium-compliant environments for real business problems:
      1. ChurnPrevention (Singtel/StarHub) — customer retention interventions
      2. PortfolioRebalancing (hedge fund) — risk-adjusted asset allocation
      3. QueueManagement (Changi Airport) — staff allocation to counters
      4. EnergyTrading (SP Group) — electricity spot market trading
      5. TrafficSignal (LTA) — green light timing optimisation
  [x] Trained DQN on ChurnPrevention and registered in ModelRegistry
  [x] Visualised environment behaviour:
      - Training reward curves showing learning progress
      - Action distribution evolution (random -> strategic)
      - Single-episode trajectory with state + action timeline
  [x] Evaluated with business metrics:
      - Churn rate reduction vs baselines
      - Revenue retained per customer per month (SGD)
      - Net value: revenue minus intervention cost
      - DQN learns SELECTIVE intervention (best net value)

  KEY INSIGHT:
  The ENVIRONMENT is the hardest part of applied RL. Get the state,
  actions, and rewards right, and even simple algorithms (DQN) learn
  useful policies. Get them wrong, and even sophisticated algorithms
  learn the wrong thing.

  Next: In 04_algorithm_comparison.py, you'll compare Random vs DQN
  vs PPO side-by-side with a decision framework for which algorithm
  to use for which problem.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Custom Environments")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `shared/mlfp05/diagnostics.py` — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.


def _diag_loss(m, batch):
    # Same as PPO/DQN — environment-specific reward
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Custom Gym Environment) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        agent,
        rollout_loader,
        _diag_loss,
        title="Custom Gym Environment",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS in range, custom env reward well-scaled.
# [?] Warning: reward magnitude 1e+3 (high) — normalise for stable TD learning.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [PRESCRIPTION] Custom environments often have poorly-scaled
#     rewards. If rewards are in the thousands, value estimates
#     explode and gradients blow up.
#     >> Prescription: normalise rewards to roughly [-1, 1] range
#        OR use reward clipping (env wrappers) OR adjust
#        discount factor gamma.
#
#  [STETHOSCOPE] Healthy gradient flow proves the environment is
#     learnable — the agent IS getting signal. Reward scaling is
#     an optimisation hygiene issue, not a design issue.

